In [1]:
import ir_datasets
import ir_datasets_owi
import os

# Tell Java inside this notebook to use Java 21
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-21-openjdk"
os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]
ir_datasets_owi.register()
dataset = ir_datasets.load("owi/dev")
dataset = ir_datasets.load("owi/test")
dataset = ir_datasets.load("owi/subsampled/dev")
dataset = ir_datasets.load("owi/subsampled/test")



In [4]:

import ir_datasets
import ir_datasets_owi
import os
# Load a dataset
dataset = ir_datasets.load("owi/subsampled/dev")

# Check if qrels exist
if hasattr(dataset, 'qrels'):
    print("✓ Qrels are available!")
    
    # Count how many qrels
    qrels_count = 0
    for qrel in dataset.qrels_iter():
        qrels_count += 1
    print(f"Total qrels: {qrels_count}")
    
    # Show first few examples
    print("\nFirst 5 qrels:")
    for i, qrel in enumerate(dataset.qrels_iter()):
        if i >= 5:
            break
        print(f"  Query ID: {qrel.query_id}, Doc ID: {qrel.doc_id}, Relevance: {qrel.relevance}")
else:
    print("✗ No qrels available for this dataset")

# Also check what's available in the dataset
print("\nAvailable components:")
print(f"  - docs: {hasattr(dataset, 'docs_iter')}")
print(f"  - queries: {hasattr(dataset, 'queries_iter')}")
print(f"  - qrels: {hasattr(dataset, 'qrels_iter')}")

# Quick view
queries = list(dataset.queries_iter())
print(f"Total queries: {len(queries)}\n")

print("First 5 queries:")
for query in queries[:5]:
    print(f"  [{query.query_id}] {query.text}")

✓ Qrels are available!
Total qrels: 1311

First 5 qrels:
  Query ID: 3, Doc ID: 0434be636200bec94cf9c9267d83a314db5ed66f7daa040586c0100bb49359a5, Relevance: 0
  Query ID: 3, Doc ID: 0613590b24db271a692b0e0eea44ce4ac48080d5c230d7071c00cc1724b1613f, Relevance: 2
  Query ID: 3, Doc ID: 1ddb8e93d2259c719bd72ea589132b4b3368e72cb2dac62b919a64d714a7b113, Relevance: 1
  Query ID: 3, Doc ID: 1f2c20f36169182cf77aa08b14657783fb2caffc48d21187bf3e47766e832e50, Relevance: 0
  Query ID: 3, Doc ID: 2a5b6bb9235b3ada7e1ee8328b3bb035303ec962ce118313a4eabff5874adce6, Relevance: 0

Available components:
  - docs: True
  - queries: True
  - qrels: True
Total queries: 28

First 5 queries:
  [3] split ergo keyboard
  [4] metoo Hollywood
  [7] gastritis symptoms
  [8] What is privacy by design(PbD)?
  [13] Impact of Exercise on Depression


In [5]:
# Get the first document
doc = next(dataset.docs_iter())
doc = next(dataset.qr())

# Print all fields (namedtuple fields)
print(doc._fields)  # if it's a namedtuple

# Or as a dictionary
print(doc._asdict())

AttributeError: qr

In [3]:
for q in dataset.queries_iter():
    print(q)


GenericQuery(query_id='1', text='american civil war')
GenericQuery(query_id='2', text='white shoes cleaning')
GenericQuery(query_id='5', text='Safest vehicles')
GenericQuery(query_id='6', text='Nautical mile')
GenericQuery(query_id='9', text='biggest church milan')
GenericQuery(query_id='10', text='best networking methods')
GenericQuery(query_id='11', text='allergy friendly cats')
GenericQuery(query_id='12', text='Hiphop dance competitions')
GenericQuery(query_id='14', text='LeBron James GOAT debate')
GenericQuery(query_id='17', text='Lung Cancer')
GenericQuery(query_id='19', text='Nuclear energy France')
GenericQuery(query_id='21', text='Gartner hype cycle')
GenericQuery(query_id='22', text='easiest filament 3d printing')
GenericQuery(query_id='25', text='Oasis famous songs')
GenericQuery(query_id='26', text='100 men versus gorilla')
GenericQuery(query_id='27', text='who is the ceo of steam')
GenericQuery(query_id='28', text='maximum particulate matter levels Europe')
GenericQuery(que

In [6]:
print("Number of docs:", dataset.docs_count())
print("Number of queries:", dataset.queries_count())


Number of docs: 357212
Number of queries: None


In [5]:

print(doc._fields)  # if it's a namedtuple

('doc_id', 'url', 'main_content', 'title', 'description')


In [16]:
import json
from pathlib import Path
import ir_datasets

docs= dataset.docs_iter()
# json file creation
# writing documents into file json
output_folder = Path("data/smallsample")
output_folder.mkdir(parents=True, exist_ok=True)
output_path = output_folder / "docs.jsonl"

with open(output_path, "w", encoding="utf-8") as f_out:
    for i,doc in enumerate(docs):
        if(doc.title != None and doc.main_content != None):
            text = (doc.title + " " + doc.main_content).strip() if hasattr(doc, "title") else doc.text
            record = {"id": doc.doc_id, "text": text}
            f_out.write(json.dumps(record, ensure_ascii=False) + "\n")
        if i==100:
            break


In [ ]:
%%bash

export JAVA_HOME=/usr/lib/jvm/java-21-openjdk
export PATH=$JAVA_HOME/bin:$PATH

python -m pyserini.index.lucene \
  --collection JsonCollection \
  --input data/owi_samplejsonl \
  --index pyserini_indexes/owi_sample_lucineindex \
  --generator DefaultLuceneDocumentGenerator \
  --threads 32 \
  --storePositions \
  --storeDocvectors \
  --storeRaw

In [17]:
%%bash

export JAVA_HOME=/usr/lib/jvm/java-21-openjdk
export PATH=$JAVA_HOME/bin:$PATH

python -m pyserini.encode \
  input \
    --corpus data/smallsample/docs.jsonl\
    --fields text \
    --shard-id 0 \
    --shard-num 1 \
  output \
    --embeddings denseindex/ \
    --to-faiss \
  encoder \
    --encoder castorini/tct_colbert-v2-hnp-msmarco \
    --fields text \
    --batch 16 \
    --fp16

100it [00:00, 34119.45it/s]
  0%|          | 0/7 [00:00<?, ?it/s]/home/luuk/Uni/IR/project/myvenv/lib/python3.10/site-packages/pyserini/encode/_tct_colbert.py:67: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
100%|██████████| 7/7 [00:00<00:00, 11.62it/s]


1. 007ff2ff733f1cad0a249cbe9745650108f2c4a5970e37e2ea4f2ea3e3b67442 (score: 77.03250885009766)
2. 007f034f1c082c2c552a4ca4f3830bdf0cf80782fb3a3043a86402558b7c541f (score: 76.87461853027344)
3. 007b089b49bb2b95f983fcb4e7f42ec02a15687cb7ff4f55a6636037d10ab308 (score: 76.7802505493164)
4. 0084efca5e63240466b97bbda5dbc941567224fc1d0b9607d94cf9f0929fc2b7 (score: 76.69387817382812)
5. 0079e29d4119b05f514fcec2be0dcf9a219208d689d0b35205e4ccff360fe3bd (score: 76.6743392944336)
6. 00783aa8d6ed21e4ca1b93207252a432935d7ea7ba756ebef841f66e1037d67f (score: 76.66619110107422)
7. 00815e81a8d16d905083a0e7ee851d1a97aae68fbf047b21dbf9a0dae7daacec (score: 76.64813232421875)
8. 007aca91e2d831a15120058d710c4f3c65289224c0db666a202c99b51a77e021 (score: 76.46350860595703)
9. 0080a9da17a08b0ce95f0cdb884cf50ed19ca54027cdcc8ed89996d3ca3d726c (score: 76.40187072753906)
10. 0083b6db15004023f156cb6d6c09030884e5675f79d75f309be4eb6c162e4113 (score: 76.39924621582031)


In [7]:
from pyserini.search.faiss import FaissSearcher

# Initialize the searcher with your index
searcher = FaissSearcher(
    'colbert_encoded_docs/',
    'castorini/tct_colbert-v2-hnp-msmarco'
)

# Perform a search
query = "How to eat lobster"
hits = searcher.search(query, k=10)  # k is the number of results to return

# Process the results
for i, hit in enumerate(hits):
    print(f"{i+1}. {hit.docid} (score: {hit.score})")
    # If you want to see the document content, you'll need to load it separately
    # from your original corpus using the docid

/home/luuk/Uni/IR/project/myvenv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


1. 8dd185b8508442cae9109dcacdce3f70a774d671155379e83b07fe572cd43358 (score: 78.77590942382812)
2. 16e1e58b04a439f0d808a14e2eb613c0df751d0165e03d4bbdc49f2c93c9b0ea (score: 78.58946228027344)
3. 64ff0c367d30a86a3b63bb0e94074eeda9e40f592fb69758a3dacdbc4c7199f7 (score: 78.47157287597656)
4. 0246333136b1c37c0e561162c4cf21698ddf0168407a135feebbb0b1b8be52dd (score: 78.4571533203125)
5. 75268d9a814974d30293b8b85d85995c50aa548bb3cb76aef5886da7f584bafb (score: 78.39229583740234)
6. 6a9c1e75d6fdbf1a3633011954a474a548e4958488b07dba6ad96f144bd761f0 (score: 78.367919921875)
7. b7f4a22530e4c0c2157a20e138f25f7aa31b80c1e40cf2babd32178302ee68c0 (score: 78.31721496582031)
8. 655cf8b20a9dbfc3f4ddc7cc3f93b945aaeb5fbb9d88965deea07d24004eec01 (score: 78.22885131835938)
9. caf530bf1ceb28518e16742ae0c2c157ff60b9ae46832001c7a7fbc0eaac9776 (score: 78.11625671386719)
10. a054ee7ce4041cb69623d38eb9d2c7dc63e2c5259a2708fd6a7539775ec25b67 (score: 77.9231185913086)


In [8]:
from pyserini.search.faiss import FaissSearcher
from pyserini.search.lucene import LuceneSearcher
from pyserini.search.hybrid import HybridSearcher

# Initialize both searchers
dense_searcher = FaissSearcher('denseindex/', 'castorini/tct_colbert-v2-hnp-msmarco')
sparse_searcher = LuceneSearcher('pyserini_indexes/owi_sample_lucineindex')

# Hybrid search combines both approaches
hybrid_searcher = HybridSearcher(dense_searcher, sparse_searcher)

query = "How to run a marathon?"
hits = hybrid_searcher.search(query, k=10, alpha=0.25)  # alpha controls the mix

for hit in hits:
    doc = sparse_searcher.doc(hit.docid)
    print(f"DocID: {hit.docid}, Score: {hit.score}")
    print(f"Text: {doc.raw()}\n")

DocID: 0080159fad69fc9ed80c85b7e116b98b5d061c14ba8abc3733bac8a1eca16929, Score: 79.5660730600357
Text: {
  "id" : "0080159fad69fc9ed80c85b7e116b98b5d061c14ba8abc3733bac8a1eca16929",
  "contents" : "20 week training program for 1/2 marathon | Experienced runners discussions | Well Being center | SteadyHealth.com <h3>Couldn't find what you looking for?</h3>\n\n<h2>TRY OUR SEARCH!</h2>\n\nMy sites are set on the Clarion River Half Marathon the first weekend in April. This is the training program I have put together to prepare for it. I ran a half in Sept. and finished in 2:02, wanna do this one in 1:50, think this can help get me there??? These first five weeks are merely a continuation of the base building I have been doing for the last few months, getting my mileage up to a respectable level.<br><br>\nSun--------Mon--------Tue-------Wed--------Thu---------Fri---------Sat<br>\nlsd 10------ez4---------6-----------R-----------6-----------R----------ez4<br>\nlsd 6-------ez4---------6-------

Dec 07, 2025 7:03:16 PM org.apache.lucene.store.MemorySegmentIndexInputProvider <init>
INFO: Using MemorySegmentIndexInput with Java 21; to disable start with -Dorg.apache.lucene.store.MMapDirectory.enableMemorySegments=false


In [36]:
from pyserini.search.lucene import LuceneSearcher

# Document retrieval
doc_searcher = LuceneSearcher('pyserini_indexes/owi_sample_lucineindex')


query = "How to run a marathon?"
hits = doc_searcher.search(query, k=10)

for hit in hits:
    doc = doc_searcher.doc(hit.docid)
    print(f"DocID: {hit.docid}, Score: {hit.score}")
    print(f"Text: {doc.raw()}\n")

DocID: ca6b71b661ab7d0afdbb034d649104d6c25d172b79c363ef43a632309b8fe794, Score: 7.915599822998047
Text: {
  "id" : "ca6b71b661ab7d0afdbb034d649104d6c25d172b79c363ef43a632309b8fe794",
  "contents" : "How to train for a marathon or half marathon <p> </p>\n\n<h2>How to Train for a <br>\nMarathon or Half Marathon<br></h2>\n\n<p> Want to learn how to train for a marathon or half marathon? Welcome to Brad Boughman's MarathonRookie.com! This site was created for the runner who wants to experience the thrill and sensational high of<br>\nfinishing their first 26.2-mile marathon or 13.1-mile half marathon. </p>\n\n<p>The secret to successful marathon training and half marathon training lies within staying supremely motivated, training smart &amp; safe, and maintaining proper nutrition. </p>\n\n<p>The days of just “gutting it out” are long gone. Whether you are training for a marathon or half marathon, it takes a lot more than good old determination and willpower to get you through training and t

In [9]:
from pyserini.search.faiss import FaissSearcher
from pyserini.search.lucene import LuceneSearcher
from pyserini.search.hybrid import HybridSearcher
import ir_datasets
import numpy as np

# Initialize searchers
dense_searcher = FaissSearcher('colbert_encoded_docs/', 'castorini/tct_colbert-v2-hnp-msmarco')
sparse_searcher = LuceneSearcher('pyserini_indexes/owi_sample_lucineindex')

# Load dataset
dataset = ir_datasets.load("owi/subsampled/dev")

# Build a mapping of query_id -> relevant doc_ids with relevance scores
qrels_dict = {}
for qrel in dataset.qrels_iter():
    if qrel.query_id not in qrels_dict:
        qrels_dict[qrel.query_id] = {}
    qrels_dict[qrel.query_id][qrel.doc_id] = qrel.relevance

def evaluate_searcher(searcher, searcher_name, k=10, alpha=None):
    """Evaluate a searcher on the dataset"""
    print(f"\n{'='*60}")
    print(f"Evaluating: {searcher_name}")
    print('='*60)
    
    total_precision = 0
    total_recall = 0
    total_mrr = 0
    total_ndcg = 0
    num_queries = 0
    
    for query in dataset.queries_iter():
        query_id = query.query_id
        query_text = query.text
        
        # Skip if no qrels for this query
        if query_id not in qrels_dict:
            continue
        
        relevant_docs = qrels_dict[query_id]
        # Consider docs with relevance > 0 as relevant (adjust as needed)
        relevant_docids = {doc_id for doc_id, rel in relevant_docs.items() if rel > 0}
        
        if len(relevant_docids) == 0:
            continue
        
        # Search
        try:
            if alpha is not None:
                # For hybrid searcher, pass alpha
                hits = searcher.search(query_text, k=k, alpha=alpha)
            else:
                hits = searcher.search(query_text, k=k)
            retrieved_docids = [hit.docid for hit in hits]
        except Exception as e:
            print(f"Error searching for query {query_id}: {e}")
            continue
        
        # Calculate metrics
        relevant_retrieved = set(retrieved_docids) & relevant_docids
        
        # Precision@K
        precision = len(relevant_retrieved) / len(retrieved_docids) if retrieved_docids else 0
        
        # Recall@K
        recall = len(relevant_retrieved) / len(relevant_docids) if relevant_docids else 0
        
        # MRR
        rr = 0
        for i, docid in enumerate(retrieved_docids, 1):
            if docid in relevant_docids:
                rr = 1 / i
                break
        
        # NDCG@K (simplified - using graded relevance)
        dcg = 0
        idcg = 0
        for i, docid in enumerate(retrieved_docids, 1):
            rel = relevant_docs.get(docid, 0)
            dcg += rel / np.log2(i + 1)
        
        # Ideal ranking (sorted by relevance)
        sorted_rels = sorted(relevant_docs.values(), reverse=True)[:k]
        for i, rel in enumerate(sorted_rels, 1):
            idcg += rel / np.log2(i + 1)
        
        ndcg = dcg / idcg if idcg > 0 else 0
        
        total_precision += precision
        total_recall += recall
        total_mrr += rr
        total_ndcg += ndcg
        num_queries += 1
    
    # Print average metrics
    if num_queries > 0:
        print(f"\nResults over {num_queries} queries:")
        print(f"  Precision@{k}: {total_precision/num_queries:.4f}")
        print(f"  Recall@{k}:    {total_recall/num_queries:.4f}")
        print(f"  MRR:           {total_mrr/num_queries:.4f}")
        print(f"  NDCG@{k}:      {total_ndcg/num_queries:.4f}")
    else:
        print("No queries evaluated!")
    
    return {
        'name': searcher_name,
        'precision': total_precision/num_queries if num_queries > 0 else 0,
        'recall': total_recall/num_queries if num_queries > 0 else 0,
        'mrr': total_mrr/num_queries if num_queries > 0 else 0,
        'ndcg': total_ndcg/num_queries if num_queries > 0 else 0
    }

# Evaluate different approaches
k = 10
results = []

# BM25 only
bm25_results = evaluate_searcher(sparse_searcher, "BM25 (Lucene)", k=k)
results.append(bm25_results)

# Dense only
dense_results = evaluate_searcher(dense_searcher, "Dense (FAISS)", k=k)
results.append(dense_results)

# Hybrid with different alpha values
for alpha in [0.1, 0.25, 0.5, 0.75, 0.9]:
    hybrid_searcher = HybridSearcher(dense_searcher, sparse_searcher)
    hybrid_results = evaluate_searcher(
        hybrid_searcher,
        f"Hybrid (α={alpha})",
        k=k,
        alpha=alpha
    )
    results.append(hybrid_results)

# Summary comparison
print(f"\n{'='*60}")
print("SUMMARY")
print('='*60)
print(f"{'Method':<20} | {'P@10':<6} | {'R@10':<6} | {'MRR':<6} | {'NDCG@10':<6}")
print("-"*60)
for result in results:
    print(f"{result['name']:<20} | {result['precision']:.4f} | {result['recall']:.4f} | {result['mrr']:.4f} | {result['ndcg']:.4f}")


Evaluating: BM25 (Lucene)

Results over 28 queries:
  Precision@10: 0.3750
  Recall@10:    0.3025
  MRR:           0.7146
  NDCG@10:      0.3971

Evaluating: Dense (FAISS)

Results over 28 queries:
  Precision@10: 0.3714
  Recall@10:    0.2857
  MRR:           0.7140
  NDCG@10:      0.3799

Evaluating: Hybrid (α=0.1)

Results over 28 queries:
  Precision@10: 0.4286
  Recall@10:    0.3581
  MRR:           0.7366
  NDCG@10:      0.4281

Evaluating: Hybrid (α=0.25)

Results over 28 queries:
  Precision@10: 0.4357
  Recall@10:    0.3608
  MRR:           0.7632
  NDCG@10:      0.4464

Evaluating: Hybrid (α=0.5)

Results over 28 queries:
  Precision@10: 0.4393
  Recall@10:    0.3658
  MRR:           0.8019
  NDCG@10:      0.4605

Evaluating: Hybrid (α=0.75)

Results over 28 queries:
  Precision@10: 0.4286
  Recall@10:    0.3550
  MRR:           0.8158
  NDCG@10:      0.4575

Evaluating: Hybrid (α=0.9)

Results over 28 queries:
  Precision@10: 0.4321
  Recall@10:    0.3567
  MRR:           0